# Pipeline End-to-End — MyCobot 280
### Rol 1: Lider de Integracion — Leon
**Problema que cubre:** P7 (Pipeline E2E)

---
**Arquitectura del sistema:**
```
vision.py          ->  detect_object(frame)    ->  (x_mm, y_mm)
cinematica.py      ->  ik_solve(x, y, z)       ->  [J1..J6]
control.py         ->  goto_pose, pick, place  ->  movimiento robot
main.py (este)     ->  maquina de estados E2E  ->  ciclo autonomo
```

**Maquina de estados:**
```
IDLE -> DETECTANDO -> CALC_IK -> AGARRANDO -> DEPOSITAR -> IDLE
```

**Orden de ejecucion:**
1. Celda 1 — Conexion al robot y camara
2. Celda 2 — Importar modulos del equipo
3. Celda 3 — Poses clave del sistema
4. Celda 4 — Logging centralizado
5. Celda 5 — Maquina de estados
6. Celda 6 — Pipeline E2E completo
7. Celda 7 — Ejecucion de 5 ciclos autonomos
8. Celda 8 — Reporte de sesion

In [ ]:
# ============================================================
# CELDA 1 — CONEXION AL ROBOT Y CAMARA
# Inicializa todos los recursos del sistema:
#   - Robot MyCobot 280 via /dev/ttyUSB0
#   - Camara via cv2.VideoCapture(0)
# ============================================================

from pymycobot.mycobot import MyCobot
import cv2
import numpy as np
import datetime
import time
import ipywidgets.widgets as widgets
from IPython.display import display

# Conexion al robot
mc = MyCobot('/dev/ttyUSB0', 1000000)
mc.power_on()
time.sleep(1)

# Verificar conexion al robot
assert mc.is_controller_connected(), "ERROR: Robot no conectado"
print("Robot conectado correctamente")

# Conexion a la camara
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

if cap.isOpened():
    print("Camara conectada correctamente")
else:
    print("ADVERTENCIA: Camara no disponible")

print("Angulos actuales:", mc.get_angles())
print("Coordenadas actuales:", mc.get_coords())

In [ ]:
# ============================================================
# CELDA 2 — MODULOS DEL EQUIPO
# Integra las funciones desarrolladas por cada integrante.
# Cada modulo fue desarrollado y verificado con el robot real.
#
# vision.py    -> Aaron  (detect_object)
# cinematica   -> Dias   (ik_solve)
# control      -> Alex   (goto_pose, pick, place)
# ============================================================

# ── MODULO VISION (Aaron) ──
# Parametros de calibracion HSV para deteccion del objeto
HSV_BAJO  = np.array([0,   120,  70])
HSV_ALTO  = np.array([10,  255, 255])
HSV_BAJO2 = np.array([160, 120,  70])
HSV_ALTO2 = np.array([180, 255, 255])
AREA_MINIMA = 500

# Parametros de transformacion pixel a mm
CX_CENTRO = 320
CY_CENTRO = 240
ESCALA_X  = 0.85
ESCALA_Y  = 0.85
OFFSET_X  =  50.0
OFFSET_Y  = -65.0

def detect_object(frame):
    """
    Modulo Vision — Aaron
    Detecta el objeto en el frame y retorna su posicion en mm.
    Retorna (x_mm, y_mm) o None si no hay objeto.
    """
    if frame is None or frame.size == 0:
        return None
    hsv   = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask1 = cv2.inRange(hsv, HSV_BAJO,  HSV_ALTO)
    mask2 = cv2.inRange(hsv, HSV_BAJO2, HSV_ALTO2)
    mask  = cv2.bitwise_or(mask1, mask2)
    kernel = np.ones((5, 5), np.uint8)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    contorno = max(contours, key=cv2.contourArea)
    if cv2.contourArea(contorno) < AREA_MINIMA:
        return None
    M  = cv2.moments(contorno)
    if M['m00'] == 0:
        return None
    cx = int(M['m10'] / M['m00'])
    cy = int(M['m01'] / M['m00'])
    x_mm = round(OFFSET_X + (cx - CX_CENTRO) * ESCALA_X, 1)
    y_mm = round(OFFSET_Y - (cy - CY_CENTRO) * ESCALA_Y, 1)
    return (x_mm, y_mm)

# ── MODULO CINEMATICA (Dias) ──
L2 = 110.4
L3 = 96.0
d1 = 131.56

def ik_solve(x, y, z):
    """
    Modulo Cinematica — Dias
    Calcula los angulos articulares para alcanzar (x, y, z).
    Retorna [J1..J6] en grados o None si no es alcanzable.
    """
    r = np.sqrt(x**2 + y**2)
    z_prima = z - d1
    distancia = np.sqrt(r**2 + z_prima**2)
    if distancia > (L2 + L3):
        return None
    theta1 = np.degrees(np.arctan2(y, x))
    cos3   = (r**2 + z_prima**2 - L2**2 - L3**2) / (2 * L2 * L3)
    cos3   = np.clip(cos3, -1, 1)
    sin3   = np.sqrt(1 - cos3**2)
    theta3 = np.degrees(np.arctan2(sin3, cos3))
    theta2 = np.degrees(
        np.arctan2(z_prima, r) -
        np.arctan2(L3 * sin3, L2 + L3 * cos3)
    )
    return [round(theta1, 2), round(theta2, 2), round(theta3, 2), 0, 0, 0]

# ── MODULO CONTROL (Alex) ──
VELOCIDAD = 20
PARAR     = False
Z_AGARRE  = 150  # altura Z fija para el agarre (mm)

def goto_pose(pose, nombre="pose", velocidad=VELOCIDAD):
    """
    Modulo Control — Alex
    Mueve el robot a una pose con verificacion de seguridad.
    """
    global PARAR
    if PARAR:
        return False
    mc.send_angles(pose, velocidad)
    time.sleep(3)
    return True

def abrir_gripper():
    mc.set_gripper_value(100, 50)
    time.sleep(1)

def cerrar_gripper():
    mc.set_gripper_value(0, 50)
    time.sleep(1)

print("Modulos cargados: detect_object, ik_solve, goto_pose")

In [ ]:
# ============================================================
# CELDA 3 — POSES CLAVE DEL SISTEMA
# Poses medidas fisicamente en el laboratorio.
# Usadas por la maquina de estados en cada ciclo.
# ============================================================

# Pose de reposo — siempre comenzar y terminar aqui
pose_inicial   = [0.7,   -0.17,  -0.61,  -1.58,  -0.35, -44.29]

# Pose de observacion — robot apunta al area de deteccion
pose_observar  = [-3.25, -60.02, -1.05, -22.06,   6.41, -44.38]

# Pose de agarre — extremo sobre el objeto detectado
pose_agarrar   = [-2.1,  -81.29, -1.23,   1.05,   6.41, -44.29]

# Pose de deposito — zona donde se deja el objeto
pose_depositar = [89.12, -72.15, -1.23,  -4.21,   4.48, -44.12]

# Mover a pose inicial al cargar
mc.send_angles(pose_inicial, VELOCIDAD)
time.sleep(3)

print("Poses cargadas")
print("pose_inicial   :", pose_inicial)
print("pose_observar  :", pose_observar)
print("pose_agarrar   :", pose_agarrar)
print("pose_depositar :", pose_depositar)

In [ ]:
# ============================================================
# CELDA 4 — LOGGING CENTRALIZADO
# Registra todas las operaciones del sistema con timestamp.
# Cada evento incluye: hora, estado, resultado.
# Se usa en todos los estados de la maquina.
# ============================================================

# Log global de la sesion
SESSION_LOG = []

def log(estado, mensaje, resultado="OK"):
    """
    Registra un evento en el log de sesion con timestamp.

    Parametros:
        estado:    nombre del estado actual de la maquina
        mensaje:   descripcion del evento
        resultado: OK, FALLO, ADVERTENCIA
    """
    hora = datetime.datetime.now().strftime('%H:%M:%S')
    entrada = f"[{hora}] [{estado:>12}] [{resultado:>11}] {mensaje}"
    SESSION_LOG.append(entrada)
    print(entrada)

def mostrar_log():
    """Imprime el log completo de la sesion."""
    print("\n" + "=" * 65)
    print("   LOG DE SESION COMPLETO")
    print("=" * 65)
    for entrada in SESSION_LOG:
        print(entrada)
    print("=" * 65)

# Botones de emergencia
boton_parar = widgets.Button(
    description='PARAR EMERGENCIA',
    button_style='danger',
    layout=widgets.Layout(width='250px', height='45px')
)
boton_continuar = widgets.Button(
    description='CONTINUAR',
    button_style='success',
    layout=widgets.Layout(width='250px', height='45px')
)
output_log = widgets.Output()

def parar_clicked(b):
    global PARAR
    PARAR = True
    mc.send_angles(pose_inicial, 20)
    log("EMERGENCIA", "Parada activada por operador", "ADVERTENCIA")

def continuar_clicked(b):
    global PARAR
    PARAR = False
    log("IDLE", "Sistema listo para continuar", "OK")

boton_parar.on_click(parar_clicked)
boton_continuar.on_click(continuar_clicked)

display(widgets.HBox([boton_parar, boton_continuar]))
print("Sistema de logging listo")

In [ ]:
# ============================================================
# CELDA 5 — MAQUINA DE ESTADOS
#
# Estados del pipeline:
#   IDLE       -> sistema en reposo, robot en pose inicial
#   DETECTANDO -> camara activa, buscando objeto
#   CALC_IK    -> calcula angulos para la posicion detectada
#   AGARRANDO  -> ejecuta secuencia de agarre
#   DEPOSITAR  -> lleva el objeto a la zona de deposito
#
# Transiciones:
#   IDLE -> DETECTANDO           (inicio de ciclo)
#   DETECTANDO -> CALC_IK        (objeto detectado)
#   DETECTANDO -> IDLE           (sin objeto, reintento)
#   CALC_IK -> AGARRANDO         (IK calculada OK)
#   CALC_IK -> DETECTANDO        (IK fallo, reintentar)
#   AGARRANDO -> DEPOSITAR       (objeto agarrado)
#   DEPOSITAR -> IDLE            (ciclo completado)
# ============================================================

# Estados posibles
ESTADOS = ['IDLE', 'DETECTANDO', 'CALC_IK', 'AGARRANDO', 'DEPOSITAR']

# Estado actual
estado_actual = 'IDLE'

def estado_IDLE():
    """
    Estado IDLE: robot en pose segura esperando inicio.
    Transicion -> DETECTANDO
    """
    log("IDLE", "Robot en pose inicial, esperando inicio")
    goto_pose(pose_inicial, "INICIAL")
    return 'DETECTANDO'

def estado_DETECTANDO(intentos=3):
    """
    Estado DETECTANDO: captura frames y busca el objeto.
    Transicion -> CALC_IK (si detecta) o IDLE (si no detecta)
    """
    log("DETECTANDO", "Buscando objeto con camara...")
    goto_pose(pose_observar, "OBSERVAR")

    for intento in range(intentos):
        ret, frame = cap.read()
        if not ret:
            log("DETECTANDO", "Error al capturar frame", "FALLO")
            continue

        resultado = detect_object(frame)

        if resultado:
            x_mm, y_mm = resultado
            log("DETECTANDO", f"Objeto detectado en x={x_mm}mm, y={y_mm}mm")
            return 'CALC_IK', (x_mm, y_mm)

        log("DETECTANDO", f"Intento {intento+1}/{intentos}: objeto no encontrado", "ADVERTENCIA")
        time.sleep(0.5)

    log("DETECTANDO", "Objeto no detectado tras todos los intentos", "FALLO")
    return 'IDLE', None

def estado_CALC_IK(posicion_mm):
    """
    Estado CALC_IK: calcula los angulos para la posicion detectada.
    Transicion -> AGARRANDO (si IK ok) o DETECTANDO (si falla)
    """
    x_mm, y_mm = posicion_mm
    log("CALC_IK", f"Calculando IK para x={x_mm}mm, y={y_mm}mm, z={Z_AGARRE}mm")

    angulos = ik_solve(x_mm, y_mm, Z_AGARRE)

    if angulos is None:
        log("CALC_IK", "Posicion no alcanzable con IK analitica", "FALLO")
        return 'DETECTANDO', None

    log("CALC_IK", f"Angulos calculados: {angulos}")
    return 'AGARRANDO', angulos

def estado_AGARRANDO(angulos):
    """
    Estado AGARRANDO: ejecuta la secuencia de agarre.
    Transicion -> DEPOSITAR
    """
    log("AGARRANDO", "Iniciando secuencia de agarre")
    abrir_gripper()
    goto_pose(angulos, "IK_POSE")     # Mover a posicion calculada por IK
    goto_pose(pose_agarrar, "AGARRAR")  # Bajar al objeto
    cerrar_gripper()
    goto_pose(pose_observar, "SUBIR")   # Subir con objeto
    log("AGARRANDO", "Objeto agarrado correctamente")
    return 'DEPOSITAR'

def estado_DEPOSITAR():
    """
    Estado DEPOSITAR: lleva el objeto a la zona de deposito.
    Transicion -> IDLE (ciclo completado)
    """
    log("DEPOSITAR", "Moviendo objeto a zona de deposito")
    goto_pose(pose_depositar, "DEPOSITAR")
    abrir_gripper()
    goto_pose(pose_inicial, "INICIAL")
    log("DEPOSITAR", "Objeto depositado. Ciclo completado")
    return 'IDLE'

print("Maquina de estados lista")
print("Estados:", ESTADOS)

In [ ]:
# ============================================================
# CELDA 6 — PIPELINE E2E COMPLETO (P7)
# Integra los 3 modulos en un ciclo autonomo completo:
#   Vision -> IK -> Control
# Ejecuta la maquina de estados hasta completar un ciclo.
# ============================================================

def ejecutar_ciclo():
    """
    Ejecuta un ciclo completo del pipeline E2E.
    Recorre la maquina de estados hasta volver a IDLE.

    Retorna:
        True  si el ciclo se completo con exito
        False si ocurrio algun fallo
    """
    global PARAR
    estado = 'IDLE'
    datos  = None

    try:
        # Estado IDLE -> DETECTANDO
        estado = estado_IDLE()
        if PARAR: return False

        # Estado DETECTANDO -> CALC_IK o IDLE
        estado, datos = estado_DETECTANDO()
        if PARAR or datos is None: return False

        # Estado CALC_IK -> AGARRANDO o DETECTANDO
        estado, angulos = estado_CALC_IK(datos)
        if PARAR or angulos is None: return False

        # Estado AGARRANDO -> DEPOSITAR
        estado = estado_AGARRANDO(angulos)
        if PARAR: return False

        # Estado DEPOSITAR -> IDLE
        estado = estado_DEPOSITAR()

        return True

    except Exception as e:
        log("ERROR", f"Excepcion en estado {estado}: {e}", "FALLO")
        goto_pose(pose_inicial, "EMERGENCIA")  # Volver a pose segura
        return False

print("Pipeline E2E listo")
print("Flujo: IDLE -> DETECTANDO -> CALC_IK -> AGARRANDO -> DEPOSITAR -> IDLE")

In [ ]:
# ============================================================
# CELDA 7 — EJECUCION DE 5 CICLOS AUTONOMOS
# Ejecuta el pipeline completo 5 veces sin intervencion humana.
# Registra: tiempo de ciclo, estado final, tasa de exito.
# ============================================================

def run_pipeline(n=5):
    """
    Ejecuta n ciclos autonomos completos del pipeline E2E.
    Registra resultado de cada ciclo en el log de sesion.
    """
    global PARAR, SESSION_LOG
    PARAR = False
    SESSION_LOG = []
    resultados = []

    log("SISTEMA", f"Iniciando sesion de {n} ciclos autonomos")
    inicio_sesion = datetime.datetime.now()

    for i in range(n):
        print("\n" + "=" * 65)
        print(f"   CICLO {i+1} de {n}")
        print("=" * 65)

        inicio_ciclo = datetime.datetime.now()
        exito = ejecutar_ciclo()
        fin_ciclo = datetime.datetime.now()
        duracion = (fin_ciclo - inicio_ciclo).seconds

        if exito:
            msg = f"Ciclo {i+1}: EXITO — {duracion}s"
        else:
            msg = f"Ciclo {i+1}: FALLO — {duracion}s"

        resultados.append(msg)
        log("SISTEMA", msg)

        # Pausa entre ciclos
        if i < n - 1:
            time.sleep(2)

    # Resumen final de la sesion
    fin_sesion = datetime.datetime.now()
    duracion_total = (fin_sesion - inicio_sesion).seconds
    exitos = sum(1 for r in resultados if "EXITO" in r)

    print("\n" + "=" * 65)
    print("   RESUMEN DE SESION")
    print("=" * 65)
    for r in resultados:
        print(r)
    print(f"\nTasa de exito:    {exitos}/{n} ({exitos/n*100:.0f}%)")
    print(f"Duracion total:   {duracion_total}s")
    print(f"Ciclos exitosos:  {exitos}")
    print(f"Ciclos fallidos:  {n - exitos}")

    log("SISTEMA", f"Sesion finalizada: {exitos}/{n} exitosos en {duracion_total}s")

# Ejecutar 5 ciclos autonomos
run_pipeline(5)

In [ ]:
# ============================================================
# CELDA 8 — REPORTE DE SESION
# Muestra el log completo de la sesion para el informe.
# Incluye todos los eventos con timestamp y estado.
# ============================================================

# Mostrar log completo
mostrar_log()

# Liberar camara al terminar
cap.release()
print("\nCamara liberada")
print("Pipeline E2E finalizado")